# Регрессия — Ridge, Lasso, LinearRegression (МНК), DecisionTree

Подбор гиперпараметров через **Optuna**.

## 0. Импорты

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set(style='whitegrid')
%matplotlib inline

## 1. Загрузка данных

👉 **Подставь свой датасет и имя таргета.**

In [ ]:
# ===== ТВОИ НАСТРОЙКИ =====
DATASET_PATH = 'your_dataset.csv'   # путь к файлу
TARGET_COL   = 'target'             # имя целевого столбца
# ===========================

df = pd.read_csv(DATASET_PATH)
print(f'Shape: {df.shape}')
df.head()

## 2. Предобработка

In [ ]:
# Удаляем строки с пропусками (простой вариант)
df = df.dropna()
print(f'After dropna: {df.shape}')

# Отделяем X и y
y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

# Оставляем только числовые признаки
X = X.select_dtypes(include=[np.number])
print(f'Features: {X.shape[1]}')

## 3. Исследовательский анализ (EDA)

👉 **Укажи список столбцов для отрисовки в `PLOT_COLS`.**

In [ ]:
# ===== СТОЛБЦЫ ДЛЯ ГРАФИКОВ =====
PLOT_COLS = list(X.columns)[:6]   # первые 6 признаков (измени по желанию)
# =================================

fig, axes = plt.subplots(1, len(PLOT_COLS), figsize=(5 * len(PLOT_COLS), 4))
if len(PLOT_COLS) == 1:
    axes = [axes]
for ax, col in zip(axes, PLOT_COLS):
    ax.hist(X[col], bins=30, edgecolor='black')
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: каждый выбранный признак vs target
fig, axes = plt.subplots(1, len(PLOT_COLS), figsize=(5 * len(PLOT_COLS), 4))
if len(PLOT_COLS) == 1:
    axes = [axes]
for ax, col in zip(axes, PLOT_COLS):
    ax.scatter(X[col], y, alpha=0.3, s=8)
    ax.set_xlabel(col)
    ax.set_ylabel(TARGET_COL)
plt.tight_layout()
plt.show()

In [ ]:
# Корреляционная матрица
corr = pd.concat([X[PLOT_COLS], y], axis=1).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

## 4. Train/Test split + масштабирование

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}')

## 5. Optuna — подбор гиперпараметров

Для каждой модели — отдельная функция цели.

In [ ]:
N_TRIALS = 50   # количество итераций Optuna (увеличь для лучшего поиска)

### 5.1 Ridge

In [ ]:
def objective_ridge(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 100.0, log=True)
    model = Ridge(alpha=alpha)
    score = cross_val_score(model, X_train_scaled, y_train,
                            cv=5, scoring='neg_mean_squared_error').mean()
    return score

study_ridge = optuna.create_study(direction='maximize')
study_ridge.optimize(objective_ridge, n_trials=N_TRIALS)
print('Ridge best params:', study_ridge.best_params)

### 5.2 Lasso

In [ ]:
def objective_lasso(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 100.0, log=True)
    model = Lasso(alpha=alpha, max_iter=10000)
    score = cross_val_score(model, X_train_scaled, y_train,
                            cv=5, scoring='neg_mean_squared_error').mean()
    return score

study_lasso = optuna.create_study(direction='maximize')
study_lasso.optimize(objective_lasso, n_trials=N_TRIALS)
print('Lasso best params:', study_lasso.best_params)

### 5.3 Linear Regression (МНК)

In [ ]:
# У МНК нет гиперпараметров — обучаем напрямую
lr = LinearRegression()
lr_cv = cross_val_score(lr, X_train_scaled, y_train,
                        cv=5, scoring='neg_mean_squared_error')
print(f'LR CV MSE: {-lr_cv.mean():.4f} ± {lr_cv.std():.4f}')

### 5.4 Decision Tree (регрессия)

In [ ]:
def objective_dt(trial):
    max_depth = trial.suggest_int('max_depth', 2, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 20)
    model = DecisionTreeRegressor(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    score = cross_val_score(model, X_train, y_train,
                            cv=5, scoring='neg_mean_squared_error').mean()
    return score

study_dt = optuna.create_study(direction='maximize')
study_dt.optimize(objective_dt, n_trials=N_TRIALS)
print('DT best params:', study_dt.best_params)

## 6. Обучение финальных моделей и метрики

In [ ]:
models = {
    'Ridge': Ridge(**study_ridge.best_params),
    'Lasso': Lasso(**study_lasso.best_params, max_iter=10000),
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(**study_dt.best_params, random_state=42),
}

results = []

for name, model in models.items():
    # Линейные модели обучаем на масштабированных данных
    if name in ('Ridge', 'Lasso', 'LinearRegression'):
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
results_df

## 7. Визуализация сравнения моделей

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    bars = ax.bar(results_df['Model'], results_df[metric], edgecolor='black')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Предсказания vs реальные значения (лучшая модель)

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = models[best_name]

if best_name in ('Ridge', 'Lasso', 'LinearRegression'):
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_best, alpha=0.4, s=12)
mn, mx = min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())
plt.plot([mn, mx], [mn, mx], 'r--', lw=2)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title(f'{best_name}: Actual vs Predicted')
plt.tight_layout()
plt.show()

## 9. Важность признаков (Decision Tree)

In [ ]:
dt_model = models['DecisionTree']
feat_imp = pd.Series(dt_model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True).tail(15)

plt.figure(figsize=(6, max(3, len(feat_imp) * 0.35)))
feat_imp.plot.barh(edgecolor='black')
plt.title('Decision Tree Feature Importances')
plt.tight_layout()
plt.show()

## 10. Коэффициенты линейных моделей

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Ridge': models['Ridge'].coef_,
    'Lasso': models['Lasso'].coef_,
    'LR': models['LinearRegression'].coef_,
})
coef_df = coef_df.set_index('Feature')
coef_df.plot.bar(figsize=(12, 5), edgecolor='black')
plt.title('Linear model coefficients')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 11. Деплой лучшей модели

In [ ]:
import joblib

# Сохраняем модель и скейлер
joblib.dump(best_model, 'best_regression_model.pkl')
if best_name in ('Ridge', 'Lasso', 'LinearRegression'):
    joblib.dump(scaler, 'regression_scaler.pkl')

print(f'Лучшая модель: {best_name}')
print(f'R2 = {results_df.iloc[0]["R2"]:.4f}')
print('Сохранено в best_regression_model.pkl')